# PRISM - GigaPath + MHIST
# Pathology Reliability In Scarce-label Medicine

In [1]:
import torch
import numpy as np
import pandas as pd
import os
from google.colab import userdata, drive
from torchvision import transforms

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/gigapath/mhist'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


In [3]:
!pip install transformers timm einops huggingface_hub scikit-learn scipy tqdm -q

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Login OK")

Login OK


In [4]:
import timm
model = timm.create_model("hf-hub:prov-gigapath/prov-gigapath", pretrained=True)
model = model.cuda().eval()
print("GigaPath loaded!")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

GigaPath loaded!


In [5]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [6]:
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image

class MHISTDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(f"{self.img_dir}/{self.df.iloc[idx]['Image Name']}").convert('RGB')
        label = 1 if self.df.iloc[idx]['Majority Vote Label'] == 'SSA' else 0
        if self.transform: img = self.transform(img)
        return img, label

DATASET_DIR = '/content/drive/MyDrive/PRISM/datasets/mhist'
ann = pd.read_csv(f'{DATASET_DIR}/annotations.csv')
train_df = ann[ann['Partition'] == 'train'].reset_index(drop=True)
test_df  = ann[ann['Partition'] == 'test'].reset_index(drop=True)
val_df   = train_df.sample(frac=0.15, random_state=42)
train_df = train_df.drop(val_df.index).reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

train_dataset = MHISTDataset(train_df, f'{DATASET_DIR}/images', transform)
val_dataset   = MHISTDataset(val_df,   f'{DATASET_DIR}/images', transform)
test_dataset  = MHISTDataset(test_df,  f'{DATASET_DIR}/images', transform)

from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

Train: 1,849
Val:   326
Test:  977


In [7]:
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(device)
            features = model(images)
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy() if hasattr(labels, 'numpy') else np.array(labels))
    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting: 100%|██████████| 8/8 [05:20<00:00, 40.02s/it]


Train: (1849, 1536)
Extracting val features...


Extracting: 100%|██████████| 2/2 [02:36<00:00, 78.05s/it]


Val: (326, 1536)
Extracting test features...


Extracting: 100%|██████████| 4/4 [02:51<00:00, 42.77s/it]

Test: (977, 1536)


In [8]:
np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy',   train_labels)
np.save(f'{EMBED_DIR}/val_features.npy',   val_features)
np.save(f'{EMBED_DIR}/val_labels.npy',     val_labels)
np.save(f'{EMBED_DIR}/test_features.npy',  test_features)
np.save(f'{EMBED_DIR}/test_labels.npy',    test_labels)
print(f"Embeddings saved to: {EMBED_DIR}")

Embeddings saved to: /content/drive/MyDrive/PRISM/embeddings/gigapath/mhist


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece / len(y_true)

def run_label_fraction(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    y_prob = clf.predict_proba(test_f)[:, 1]
    y_pred = clf.predict(test_f)
    return {
        'model': 'GigaPath', 'dataset': 'MHIST',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_l, y_prob),
        'f1':    f1_score(test_l, y_pred),
        'brier': brier_score_loss(test_l, y_prob),
        'ece':   compute_ece(test_l, y_prob),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.6532 | ECE: 0.1433 | Brier: 0.2498
  1% | seed 123 | AUROC: 0.6738 | ECE: 0.1571 | Brier: 0.2343
  1% | seed 456 | AUROC: 0.6449 | ECE: 0.2010 | Brier: 0.2618
  5% | seed 42 | AUROC: 0.8380 | ECE: 0.1647 | Brier: 0.2061
  5% | seed 123 | AUROC: 0.8389 | ECE: 0.2020 | Brier: 0.2186
  5% | seed 456 | AUROC: 0.8366 | ECE: 0.1658 | Brier: 0.2062
  10% | seed 42 | AUROC: 0.8450 | ECE: 0.1422 | Brier: 0.1864
  10% | seed 123 | AUROC: 0.8425 | ECE: 0.1572 | Brier: 0.1902
  10% | seed 456 | AUROC: 0.8432 | ECE: 0.1650 | Brier: 0.2027
  25% | seed 42 | AUROC: 0.8721 | ECE: 0.1316 | Brier: 0.1657
  25% | seed 123 | AUROC: 0.8550 | ECE: 0.1123 | Brier: 0.1676
  25% | seed 456 | AUROC: 0.8535 | ECE: 0.1107 | Brier: 0.1680
  50% | seed 42 | AUROC: 0.8783 | ECE: 0.1071 | Brier: 0.1533
  50% | seed 123 | AUROC: 0.8691 | ECE: 0.0975 | Brier: 0.1541
  50% | seed 456 | AUROC: 0.8766 | ECE: 0.1076 | Brier: 0.1527
  100% | seed 42 | AUROC: 0.8868 | ECE: 0.0827 | Brier: 0.1409
  1

In [10]:
from scipy.optimize import minimize_scalar

def find_temperature(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    val_logits = clf.decision_function(val_f).reshape(-1,1)
    val_logits = np.hstack([-val_logits, val_logits])
    T          = find_temperature(val_logits, val_l)
    test_prob_raw = clf.predict_proba(test_f)[:, 1]
    ece_raw       = compute_ece(test_l, test_prob_raw)
    test_logits = clf.decision_function(test_f).reshape(-1,1)
    test_logits = np.hstack([-test_logits, test_logits])
    scaled      = test_logits / T
    exp_s       = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s = (exp_s / exp_s.sum(axis=1, keepdims=True))[:, 1]
    ece_scaled  = compute_ece(test_l, test_prob_s)
    return {
        'model': 'GigaPath', 'dataset': 'MHIST',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc': roc_auc_score(test_l, test_prob_raw),
        'brier_raw': brier_score_loss(test_l, test_prob_raw),
        'brier_scaled': brier_score_loss(test_l, test_prob_s),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           3.1138   0.1672      0.0640           0.1031
0.05           1.9050   0.1775      0.1830          -0.0056
0.10           1.6671   0.1548      0.1512           0.0036
0.25           1.3723   0.1182      0.0992           0.0189
0.50           1.3580   0.1041      0.0765           0.0276
1.00           1.3553   0.0827      0.0536           0.0291


In [11]:
df.to_csv(f'{RESULTS_DIR}/gigapath_mhist_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/gigapath_mhist_temperature_scaling.csv', index=False)
print("Results saved!")
print(f"  {RESULTS_DIR}/gigapath_mhist_results.csv")
print(f"  {RESULTS_DIR}/gigapath_mhist_temperature_scaling.csv")

Results saved!
  /content/drive/MyDrive/PRISM/results/gigapath_mhist_results.csv
  /content/drive/MyDrive/PRISM/results/gigapath_mhist_temperature_scaling.csv
